In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import time
from smbus2 import SMBus,i2c_msg
import MCP342x
from scservo_sdk import *                    # Uses SCServo SDK library
import MCP342x
import logging
from pts_iterator import pts_iterator

logging.basicConfig(
    # level=logging.INFO,  # Set the logging level to DEBUG
    level= logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

from servodriver import Servoset
from servo_util import create_zigzag_X, format_para,a2p,r2nd,r2nr,ndmodr,nrselr,nrmodr,nraddr,compose_para
from spiral import SpiralPath
from scipy.optimize import differential_evolution
from fit_gaussian import gaussian_2d,fit_and_plot,fit_gaussian_2d
from motor_2d_scan import motor_2d_scan
from servo_const import A_X_XDOT_MASK,A_Y_YDOT_MASK,A_X_Y_MASK,A_XDOT_YDOT_MASK,A_POS_ALL_MASK,B_X_XDOT_MASK,B_Y_YDOT_MASK,B_X_Y_MASK,B_XDOT_YDOT_MASK,B_POS_ALL_MASK,POS_ALL_MASK
from step_optimize import step_optimize
from pd import MCP3424_fiber


2024-09-24 18:10:58,693 - DEBUG - Convert 0x68 config: 0b10001000


0.000375


In [2]:
servos = Servoset(board_id=1,servo_channel_list=[0])
servos.torques_enable()
servos.set_angle([4096])
# time.sleep(2)
# servos.set_angle([2048])
servos.home()
# MCP3424_fiber.convert_and_read()

2024-09-24 18:10:58,767 - INFO - The device /dev/ttyUSB0 is not available, try next ...
2024-09-24 18:10:58,790 - INFO - Succeeded to open the port
2024-09-24 18:10:58,814 - INFO - Succeeded to change the baudrate
2024-09-24 18:10:58,819 - INFO - Servo 1x: SCServo acc set!
2024-09-24 18:10:58,824 - INFO - Servo 1x: SCServo speed set!
2024-09-24 18:10:58,830 - INFO - Servo 1x: SCServo acc set!
2024-09-24 18:10:58,834 - INFO - Servo 1x: SCServo speed set!
2024-09-24 18:10:58,839 - INFO - loading position from disk
2024-09-24 18:10:58,840 - INFO - Loaded, the angles are: 
Servo 1x: 0.0 deg	
2024-09-24 18:10:58,851 - DEBUG - Start Position: 	[ID:001] Goal:8096 Pres:2048	
2024-09-24 18:10:58,857 - INFO - [TxRxResult] Protocol does not support this function!
2024-09-24 18:10:58,857 - ERROR - [ID:001] groupSyncRead getdata failed
2024-09-24 18:10:58,858 - DEBUG - Iteration: 0	[ID:001] Goal:8096 Pres:2048	
2024-09-24 18:10:58,859 - DEBUG - [0]
2024-09-24 18:10:58,863 - INFO - [TxRxResult] Prot

In [6]:
goal_position_list = [2048,2048,2048,2048,2048,2048,2048,2048]
print(goal_position_list)
servos.set_angle(goal_position_list)

2024-09-24 12:33:04,259 - INFO - Servo 1x: SCServo zero set!
2024-09-24 12:33:04,268 - INFO - Servo 1y: SCServo zero set!
2024-09-24 12:33:04,278 - INFO - Servo 2x: SCServo zero set!
2024-09-24 12:33:04,287 - INFO - Servo 2y: SCServo zero set!
2024-09-24 12:33:04,322 - INFO - [TxRxResult] There is no status packet!
2024-09-24 12:33:04,330 - ERROR - Servo 3x: Read not sucessful!
2024-09-24 12:33:04,336 - ERROR - Servo 3x: Read not sucessful!
2024-09-24 12:33:04,342 - ERROR - Servo 3x: Read not sucessful!
2024-09-24 12:33:04,348 - ERROR - Servo 3x: Read not sucessful!
2024-09-24 12:33:04,354 - ERROR - Servo 3x: Read not sucessful!
2024-09-24 12:33:04,360 - ERROR - Servo 3x: Read not sucessful!
2024-09-24 12:33:04,396 - INFO - Servo 3x: result [TxRxResult] Incorrect status packet!
2024-09-24 12:33:04,396 - INFO - Servo 3x: SCServo zero set!
2024-09-24 12:33:04,406 - INFO - Servo 3y: SCServo zero set!
2024-09-24 12:33:04,415 - INFO - Servo 4x: SCServo zero set!
2024-09-24 12:33:04,420 - ER

In [7]:
def cord_pm_offset(N,normd,i):
    # choose a position in offset to be +-30, each sample 5 times
    j = i%4
    pm = 1 if i//4%2==0 else -1
    offset = np.zeros(4)
    offset[j] = pm*normd
    return offset

N=15
normd = 360
for i in range(N):
    goal_position_list = [2048,2048,2048,2048,2048,2048,2048,2048]
    print(goal_position_list)
    servos.set_angle(goal_position_list)
    goal_position_list = [0,2048,2048,2048,2048,2048,2048,2048]
    servos.set_angle(goal_position_list)
    # servos.home()
    # a = MCP3424_fiber.convert_and_read()
    # print('a:',a)

[2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048]
[2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048]
[2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048]
[2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048]
[2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048]
[2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048]


KeyboardInterrupt: 

In [3]:
def callback_func(para,
                  pos_mask,
                  zero=None,
                  jac=None,
                  jac_master_mask=None,
                  debug=False,
                  **kwargs):

    para_nr_move = compose_para(para, pos_mask, zero, jac, jac_master_mask,debug=debug,**kwargs)
    #
    if not debug:
        goal_position_list  = r2nd(list(para_nr_move))
        # print(goal_position_list)
        servos.set_angle(goal_position_list)
        #
        data_cache = []
        for m in range(2):
            # data = ADS1115_fiber.value
            data = MCP3424_fiber.convert_and_read()
            data_cache.append(data)
        z = float(np.mean(np.array(data)))
        # print(para,z)
        return tuple(para),z

cf0 = lambda para: callback_func(para, pos_mask=POS_ALL_MASK)
cf0([0,0,0,0,0,0,0,0])

2024-09-23 15:55:21,261 - DEBUG - Start Position: 	[ID:001] Goal:2048 Pres:2048	[ID:002] Goal:2048 Pres:2048	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048 Pres:2048	[ID:005] Goal:2048 Pres:6144	[ID:006] Goal:2048 Pres:6144	[ID:007] Goal:2048 Pres:2048	[ID:008] Goal:2048 Pres:2048	
2024-09-23 15:55:21,276 - DEBUG - Iteration: 0	[ID:001] Goal:2048 Pres:2048	[ID:002] Goal:2048 Pres:2048	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048 Pres:2048	[ID:005] Goal:2048 Pres:6144	[ID:006] Goal:2048 Pres:6144	[ID:007] Goal:2048 Pres:2048	[ID:008] Goal:2048 Pres:2048	
2024-09-23 15:55:22,587 - DEBUG - Iteration: 100	[ID:001] Goal:2048 Pres:2048	[ID:002] Goal:2048 Pres:2048	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048 Pres:2048	[ID:005] Goal:2048 Pres:6144	[ID:006] Goal:2048 Pres:6144	[ID:007] Goal:2048 Pres:2048	[ID:008] Goal:2048 Pres:2048	
2024-09-23 15:55:23,896 - DEBUG - Iteration: 200	[ID:001] Goal:2048 Pres:2048	[ID:002] Goal:2048 Pres:2048	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048

((0, 0, 0, 0, 0, 0, 0, 0), 0.4205625)

## 2D Scan

In [4]:
jac_pm_30_1 = np.load("/home/rydpiservo/servodata/jac_pm_30_1.npy")
jac_pm_50 = np.load("/home/rydpiservo/servodata/jac_pm_50.npy")
jac_pm_70 = np.load("/home/rydpiservo/servodata/jac_pm_70.npy")
jac_pm_110  = np.load("/home/rydpiservo/servodata/jac_pm_110.npy")
jac_pm_150 = np.load("/home/rydpiservo/servodata/jac_pm_150.npy")
jac_pm_190 = np.load("/home/rydpiservo/servodata/jac_pm_190.npy")

In [5]:
callback_func([30,0], pos_mask=A_X_Y_MASK, jac=jac_pm_190, jac_master_mask=A_POS_ALL_MASK,debug=False)

2024-09-23 15:55:25,363 - DEBUG - Start Position: 	[ID:001] Goal:2389 Pres:2048	[ID:002] Goal:2048 Pres:2049	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048 Pres:2048	[ID:005] Goal:973 Pres:6144	[ID:006] Goal:1958 Pres:6144	[ID:007] Goal:1325 Pres:2048	[ID:008] Goal:2133 Pres:2048	
2024-09-23 15:55:25,378 - DEBUG - Iteration: 0	[ID:001] Goal:2389 Pres:2048	[ID:002] Goal:2048 Pres:2049	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048 Pres:2048	[ID:005] Goal:973 Pres:6144	[ID:006] Goal:1958 Pres:6144	[ID:007] Goal:1325 Pres:2048	[ID:008] Goal:2133 Pres:2048	
2024-09-23 15:55:26,681 - DEBUG - Iteration: 100	[ID:001] Goal:2389 Pres:2389	[ID:002] Goal:2048 Pres:2048	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048 Pres:2048	[ID:005] Goal:973 Pres:5069	[ID:006] Goal:1958 Pres:6054	[ID:007] Goal:1325 Pres:1323	[ID:008] Goal:2133 Pres:2133	
2024-09-23 15:55:27,996 - DEBUG - Iteration: 200	[ID:001] Goal:2389 Pres:2389	[ID:002] Goal:2048 Pres:2048	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048 Pr

((30, 0), 0.41350000000000003)

In [6]:
N_pts = 12
SCAN_RANGE = 190
POS_MASK = A_X_Y_MASK
cf = lambda para: callback_func(para, pos_mask=POS_MASK, jac=jac_pm_190, jac_master_mask=A_POS_ALL_MASK)
# cf = lambda para: callback_func(para, pos_mask=POS_MASK, jac=np.eye(4), jac_master_mask=A_POS_ALL_MASK)
X,Y,Z = motor_2d_scan(N_pts,SCAN_RANGE,servos,cf)

2024-09-23 15:55:29,480 - DEBUG - Start Position: 	[ID:001] Goal:2048 Pres:2389	[ID:002] Goal:2048 Pres:2048	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048 Pres:2048	[ID:005] Goal:2048 Pres:5069	[ID:006] Goal:2048 Pres:6054	[ID:007] Goal:2048 Pres:1325	[ID:008] Goal:2048 Pres:2133	
2024-09-23 15:55:29,493 - DEBUG - Iteration: 0	[ID:001] Goal:2048 Pres:2389	[ID:002] Goal:2048 Pres:2048	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048 Pres:2048	[ID:005] Goal:2048 Pres:5069	[ID:006] Goal:2048 Pres:6054	[ID:007] Goal:2048 Pres:1325	[ID:008] Goal:2048 Pres:2133	
2024-09-23 15:55:30,801 - DEBUG - Iteration: 100	[ID:001] Goal:2048 Pres:2048	[ID:002] Goal:2048 Pres:2048	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048 Pres:2048	[ID:005] Goal:2048 Pres:6144	[ID:006] Goal:2048 Pres:6144	[ID:007] Goal:2048 Pres:2049	[ID:008] Goal:2048 Pres:2048	
2024-09-23 15:55:32,109 - DEBUG - Iteration: 200	[ID:001] Goal:2048 Pres:2048	[ID:002] Goal:2048 Pres:2048	[ID:003] Goal:2048 Pres:2048	[ID:004] Goal:2048

In [12]:


def statistics_for_gaussian2d(xdata, ydata, Idata):
    # Flatten the arrays in case they are not 1D
    xdata = xdata.flatten()
    ydata = ydata.flatten()
    Idata = Idata.flatten()

    # Calculate the sum of the data values
    sum_I = np.sum(Idata)

    # Calculate the weighted mean (mu)
    mu_x = np.sum(xdata * Idata) / sum_I
    mu_y = np.sum(ydata * Idata) / sum_I
    mu = np.array([mu_x, mu_y])

    # Center the coordinates by subtracting the mean
    x_centered = xdata - mu_x
    y_centered = ydata - mu_y

    # Calculate the elements of the covariance matrix
    sigma_xx = np.sum(Idata * x_centered * x_centered) / sum_I
    sigma_xy = np.sum(Idata * x_centered * y_centered) / sum_I
    sigma_yy = np.sum(Idata * y_centered * y_centered) / sum_I

    # Assemble the covariance matrix (cov)
    cov = np.array([[sigma_xx, sigma_xy],
                    [sigma_xy, sigma_yy]])

    return mu, cov


def fit_gaussian_2d(X,Y,Z,p0=None):
    X = np.array(X)
    Y = np.array(Y)
    Z = np.array(Z)
    # fit to gaussian_2d_cov using least square
    def _residuals(p, x, y, z):
        mu = p[:2]
        cov = np.array([[p[2], p[3]], [p[3], p[4]]])
        z_fit = np.array([gaussian_2d(x_, y_, mu, cov) for x_, y_ in zip(x, y)])
        return z - z_fit
    
    xdata = np.array(X).flatten()
    ydata = np.array(Y).flatten()
    zdata = np.array(Z).flatten()
    
    if p0 is None:
        mu, cov = statistics_for_gaussian2d(X, Y, Z)
        p0 = np.array([mu[0], mu[1], cov[0, 0], cov[0, 1], cov[1, 1]])
        print("Initial guess for p0: ", p0)

    from scipy.optimize import least_squares

    res = least_squares(_residuals, p0, args=(xdata, ydata, zdata))
    popt = res.x
    return popt


def fit_and_plot(X,Y,Z,p0=None):
    popt = fit_gaussian_2d(X,Y,Z,p0=p0)
    bounds_x = (np.min(X),np.max(X))
    bounds_y = (np.min(Y),np.max(Y))
    X_new = np.linspace(*bounds_x,100)
    Y_new = np.linspace(*bounds_y,100)
    X_new,Y_new = np.meshgrid(X_new,Y_new)
    Z_new = np.array([gaussian_2d(x_,y_,popt[:2],np.array([[popt[2],popt[3]],[popt[3],popt[4]]]) ) for x_,y_ in zip(X_new.ravel(),Y_new.ravel())])
    #
    plt.imshow(Z.reshape(X.shape)/np.max(Z),origin="lower",extent=[bounds_x[0],bounds_x[1],bounds_y[0],bounds_y[1]])
    plt.contour(X_new,Y_new,Z_new.reshape(X_new.shape),cmap="jet")
    return popt

In [ ]:
popt=fit_and_plot(X,Y,Z)

In [ ]:
cov = np.array([[popt[2],popt[3]],[popt[3],popt[4]]])
eigvals, eigvecs = np.linalg.eig(cov)
np.sqrt(eigvals)

jac_30 = 48, 58

jac_70 = 48, 70

jac_110 = 85, 113

jac_190 = 102, 120.8

In [13]:
def gaussian_2d_cov(x: float, y: float, mu, cov) -> float:
    inv_cov = np.linalg.inv(cov)
    det_cov = float(np.linalg.det(cov))
    r = np.array([x, y]).T - mu
    z = np.exp(-0.5 * (r @ inv_cov @ r.T))
    coeff = 1 / (2 * np.pi * np.sqrt(det_cov))
    return coeff * z


def guess_p0_for_gaussian2d(xdata, ydata, Idata):
    # Flatten the arrays in case they are not 1D
    xdata = xdata.flatten()
    ydata = ydata.flatten()
    Idata = Idata.flatten()

    # Calculate the sum of the data values
    sum_I = np.sum(Idata)

    # Calculate the weighted mean (mu)
    mu_x = np.sum(xdata * Idata) / sum_I
    mu_y = np.sum(ydata * Idata) / sum_I
    mu = np.array([mu_x, mu_y])

    # Center the coordinates by subtracting the mean
    x_centered = xdata - mu_x
    y_centered = ydata - mu_y

    # Calculate the elements of the covariance matrix
    sigma_xx = np.sqrt(np.sum(Idata * x_centered * x_centered) / sum_I)
    sigma_xy = np.sqrt(np.sum(Idata * x_centered * y_centered) / sum_I)
    sigma_yy = np.sqrt(np.sum(Idata * y_centered * y_centered) / sum_I)

    # Assemble the covariance matrix (cov)
    cov = np.array([[sigma_xx, sigma_xy],
                    [sigma_xy, sigma_yy]])

    return mu, cov

In [ ]:
def find_max_pos(Z):
    idx = np.argmax(Z)
    idx_i, idx_j = np.unravel_index(idx, Z.shape)
    return idx_i,idx_j
idx_x_max,idx_y_max = find_max_pos(Z)
print(f"Max position: {X[idx_x_max,idx_y_max],Y[idx_x_max,idx_y_max]}")
x_max = X[idx_x_max,idx_y_max]
y_max = Y[idx_x_max,idx_y_max]
px, py = a2p(X[idx_x_max,idx_y_max]), a2p(Y[idx_x_max,idx_y_max])
#
# servos.set_angle([px,a2p(0),py,a2p(0)])
pos_list = r2nd([x_max,y_max],POS_MASK)
servos.set_angle(pos_list)
# print(ADS1115_fiber.value)
print(MCP3424_fiber.convert_and_read())

## Iterative Spiral Optimization

In [20]:
zero = np.array([0,0,0,0,0,0,0,0],dtype=float)

In [ ]:
spiral_params = {
    'I_meaningful': 0.01,
    'D': 2,
    'SPIRAL_RESOLUTION': 15,
    'SPIRAL_SPAN': 8,
    'SINGLE_SPIRAL_SPAN': 4,
    'N_LOOPS_BEFORE_RESET_ORIGIN': 0.5,
    'MAX_X0Y0_DISPLACEMENT': 10,
    'COEF_I_RESET_ORIGIN': 1.4,
    'alpha': 0.03,
    'COEF_I_DECAY': 0.995
}
BFGS_params = {"disp": True, "maxiter": 10,  "eps": 5}
logging.getLogger().setLevel(logging.INFO)
#
bounds_single = (-100,100)

def iterative_optimize(pos_mask, method="spiral",zero=None,offset=None,offset_mask=None)->np.ndarray:
    if method == 'L-BFGS-B':
        servos.set_precision(1)
        options = BFGS_params
    elif method == 'spiral':
        servos.set_precision(5)
        options = spiral_params
    #
    N_var = np.sum(pos_mask)
    p0 = np.zeros(N_var)
    bounds = [bounds_single for i in range(N_var)]
    cf = lambda x: callback_func(x,pos_mask,zero=zero,offset=offset,offset_mask=offset_mask)
    #
    para, Ibst = pts_iterator(N_var=N_var,callback_func=cf ,p0=p0, bounds = bounds, options=options, method = method)
    #
    para=list(para)
    Inow = cf(para)[1]
    logging.info(f"Best position: {format_para(para)}, now I: {Inow}")
    #
    if Inow/Ibst > 0.8:
        logging.info(f"New Origin set to be {format_para(para)}")
        zero_fullnd = nraddr(zero,para,pos_mask)
    else:
        logging.info("The intensity is not high enough, operation cancelled.")
        zero_fullnd = np.array(zero)
    #
    logging.info(f"Zero = {zero_fullnd}")
    return zero_fullnd

try:
    logging.info(f"Start optimization with zero = {zero}")
    # zero = iterative_optimize(B_X_Y_MASK, zero=zero,offset=offset,offset_mask=A_POS_ALL_MASK)
    # zero = iterative_optimize(B_X_XDOT_MASK, zero=zero,offset=offset,offset_mask=A_POS_ALL_MASK)
    # zero = iterative_optimize(B_Y_YDOT_MASK, zero=zero,offset=offset,offset_mask=A_POS_ALL_MASK)
    # zero = iterative_optimize(B_POS_ALL_MASK, method='L-BFGS-B',zero=zero,offset=offset,offset_mask=A_POS_ALL_MASK)
    # # #
    # zero = iterative_optimize(A_X_Y_MASK, zero=zero,offset=offset,offset_mask=B_POS_ALL_MASK)
    # zero = iterative_optimize(A_X_XDOT_MASK, zero=zero,offset=offset,offset_mask=B_POS_ALL_MASK)
    # zero = iterative_optimize(A_Y_YDOT_MASK, zero=zero,offset=offset,offset_mask=B_POS_ALL_MASK)
    # zero = iterative_optimize(A_POS_ALL_MASK, method='L-BFGS-B',zero=zero,offset=offset,offset_mask=B_POS_ALL_MASK)

    zero = iterative_optimize(POS_ALL_MASK, method='L-BFGS-B',zero=zero)

except Exception as e:
    # servos.home()
    servos.close()
    print(e)

In [ ]:
para_nr_move = nrmodr(zero,offset,r_mask=A_POS_ALL_MASK)
goal_position_list  = r2nd(para_nr_move,r_mask=POS_ALL_MASK)
print(goal_position_list)
servos.set_angle(goal_position_list)
I = MCP3424_fiber.convert_and_read()
print(I)

In [107]:
dataset.append([list(offset),list(zero),I])

In [113]:
np.save(filename,dataset)

In [112]:
filename='/home/rydpiservo/servodata/jacobian_pm_30_1.npy'
from collections import defaultdict
dataset=defaultdict(list)

In [ ]:
dataset = np.load(filename,allow_pickle=True)
dataset = dataset.item()
print(dataset)

In [93]:
servos.home()

In [ ]:
servos.set_zero()